# Data Cleaning - CDC Natality
This notebook prepares the CDC Natality fixed-width file for the conformal
prediction experiments, focused on the **pre(term) × smoking intensity**
evaluation axis (used both for Mondrian calibration and for global-calibration
per-group evaluation).

It also restores `preterm_x_smoking_intensity` as a persisted column (8
explicit groups, no automatic merging of small groups), instead of only
building it ad-hoc inside the diagnostics printout.Everything needed is derived locally from the raw underlying fields, so every
threshold used in this notebook is visible in this notebook.

## 1. Imports and configuration

This cell only sets up paths.
- `BIRTH_WEIGHT_MIN/MAX` and `MOTHER_AGE_MIN` are CDC **edited** ranges (the
  file is already guaranteed to respect them; filtering on them mainly
  removes the CDC "unknown" sentinel values, which fall outside these
  ranges).
- `BMI_MIN/MAX` are the official CDC BMI category bounds (`13.0`–`69.9`,
  matching `BMI_R`). Per the User Guide, the published file should never
  contain a value outside this range other than the `99.9` sentinel. These
  bounds are used **only as an integrity assertion** on our own parsing,
  never to edit or drop a BMI value reported by CDC.

In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ============================================================
# Configuration
# ============================================================
"must point to the raw CDC fixed-width `.txt` file."
FILE_PATH = "Birth2022(1000k).txt"

'cleaned dataset will be written.'
OUTPUT_CSV = "natality2022_clean_preterm_smoking.csv"
OUTPUT_FILTER_LOG = None  # if None, derived automatically from OUTPUT_CSV

# CDC edited ranges / accepted codes used in this project (see User Guide).
BIRTH_WEIGHT_MIN, BIRTH_WEIGHT_MAX = 227, 8165   # DBWT, grams
MOTHER_AGE_MIN = 10                         # MAGER, years

# Official CDC BMI category bounds (matches BMI_R). The User Guide
# guarantees the published file never contains BMI outside this range
# (besides the 99.9 unknown sentinel) -- used here only as an integrity
# assertion on our own parsing, never to edit or drop CDC-reported values.
BMI_MIN, BMI_MAX = 13.0, 69.9

## 2. Selected CDC columns, with readable names

Only the fields actually used downstream are read. CDC. The original CDC
field name from the User Guide is kept as an inline comment, so the mapping
to the official documentation is always visible next to the code.

Column positions are converted from the 1-indexed inclusive positions in
the 2023 Natality Public Use File User Guide to Python's 0-indexed,
end-exclusive `colspecs` format.

In [2]:
# ============================================================
# CDC fixed-width column positions
# ============================================================
COLSPECS = [
    (74, 76),    # MAGER        Mother's single years of age
    (123, 124),  # MEDUC        Mother's education (raw code 1-8, 9=unknown)

    (170, 172),  # PRIORLIVE    Prior births now living
    (172, 174),  # PRIORDEAD    Prior births now dead
    (174, 176),  # PRIORTERM    Prior other terminations
    (178, 179),  # LBO_REC      Live birth order recode (1-7,8,9)
    (181, 182),  # TBO_REC      Total birth order recode (1-7,8,9)

    (197, 200),  # ILLB_R       Interval since last live birth (months, raw)
    (213, 216),  # ILP_R        Interval since last pregnancy (months, raw)

    (223, 225),  # PRECARE      Month prenatal care began (00-10, 99=unknown)
    (237, 239),  # PREVIS       Number of prenatal visits (00-98, 99=unknown)
    (250, 251),  # WIC          WIC food during pregnancy (Y/N/U)

    (252, 254),  # CIG_0        Cigarettes before pregnancy
    (254, 256),  # CIG_1        Cigarettes 1st trimester
    (256, 258),  # CIG_2        Cigarettes 2nd trimester
    (258, 260),  # CIG_3        Cigarettes 3rd trimester

    (279, 281),  # M_Ht_In      Mother's height, total inches
    (282, 286),  # BMI          Official CDC pre-pregnancy BMI (13.0-69.9, 99.9=unknown)
    (291, 294),  # PWgt_R       Pre-pregnancy weight, pounds
    (303, 305),  # WTGAIN       Weight gain, pounds (00-97, 98=98+, 99=unknown)

    (312, 313),  # RF_PDIAB     Pre-pregnancy diabetes
    (313, 314),  # RF_GDIAB     Gestational diabetes
    (314, 315),  # RF_PHYPE     Pre-pregnancy hypertension
    (315, 316),  # RF_GHYPE     Gestational hypertension
    (316, 317),  # RF_EHYPE     Hypertension eclampsia
    (317, 318),  # RF_PPTERM    Previous preterm birth
    (324, 325),  # RF_INFTR     Infertility treatment used
    (330, 331),  # RF_CESAR     Previous cesarean
    (331, 333),  # RF_CESARN    Number of previous cesareans (00-30, 99=unknown)

    (453, 454),  # DPLURAL      Plurality recode (1=single) -> Tell if they are twins.
    (474, 475),  # SEX          Sex of infant (M/F)
    (489, 491),  # COMBGEST     Combined gestation, weeks (17-47, 99=unknown)
    (503, 507),  # DBWT         Birth weight, grams (0227-8165, 9999=unknown)
]

NAMES = [
    "mother_age", "mother_education_code",
    "prior_live_births", "prior_dead_births",
    "prior_other_terminations", "live_birth_order", "total_birth_order",
    "interval_since_last_live_birth_raw", "interval_since_last_pregnancy_raw",
    "prenatal_care_start_month", "prenatal_visits_count", "wic_received",
    "cigarettes_before_pregnancy", "cigarettes_trimester_1", "cigarettes_trimester_2", "cigarettes_trimester_3",
    "mother_height_inches", "official_cdc_bmi", "pre_pregnancy_weight_lb", "gestational_weight_gain_lb",
    "pre_pregnancy_diabetes", "gestational_diabetes", "pre_pregnancy_hypertension",
    "gestational_hypertension", "eclampsia", "previous_preterm_birth",
    "infertility_treatment", "previous_cesarean", "previous_cesarean_count",
    "plurality_code", "infant_sex", "gestational_age_weeks", "birth_weight_g",
]

PREGNANCY_CIG_COLS = ["cigarettes_trimester_1", "cigarettes_trimester_2", "cigarettes_trimester_3"]
RISK_YN_COLS = [
    "pre_pregnancy_diabetes", "gestational_diabetes", "pre_pregnancy_hypertension",
    "gestational_hypertension", "eclampsia", "previous_preterm_birth",
    "infertility_treatment", "previous_cesarean",
]

## 3. Reading the fixed-width file and parsing BMI

`read_raw_birth_file` just reads the columns above (with the readable names)
as strings and strips whitespace.

`parse_cdc_bmi` parses the official CDC BMI field. It only converts the
sentinel "unknown" codes to `NaN`; it never overwrites an out-of-range
value. Range checking happens later, as an **assertion**, not as silent
data editing (see step 4.1).


In [3]:
# ============================================================
# Reading and BMI parsing
# ============================================================
def read_raw_birth_file(path: str) -> pd.DataFrame:
    """Read the selected fixed-width columns from the CDC Natality file."""
    df = pd.read_fwf(path, colspecs=COLSPECS, names=NAMES, dtype=str)
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    return df


def parse_cdc_bmi(series: pd.Series) -> pd.Series:
    """Parse the official CDC BMI field.

    Only converts sentinel "unknown" codes to NaN. Never overwrites an
    out-of-range value -- that case is handled by an assertion in step 4.1,
    not by silent editing here.
    """
    s = series.astype(str).str.strip()
    out = pd.to_numeric(s, errors="coerce")

    # Sentinel "unknown" codes seen in this field across CDC file variants.
    out.loc[out.isin([99.9, 999, 9999])] = np.nan
    
    return out

## 4. Converting numeric columns

Everything except the Y/N/U risk fields, `infant_sex`, and `wic_received` is
converted to numeric. The BMI column is parsed separately using
`parse_cdc_bmi` defined above, overwriting `official_cdc_bmi` in place with
the parsed numeric value.

**BMI integrity check**

This is the corrected design from the previous version. Instead of a
diagnostic "implausible" flag that silently treated CDC's own published
data as suspect, this is an **assertion**: per the User Guide, the
published file should never contain `official_cdc_bmi` outside
`13.0–69.9` (besides the `99.9` sentinel, already converted to `NaN`).

If this assertion ever fails, it means **our own pipeline** has a bug —
most likely a wrong column offset for the BMI field, or a decimal-point
parsing issue (see step 3.1) — not a genuine CDC data quality problem.
Run this right after `to_numeric_columns`, before trusting anything
downstream.

In [4]:
def to_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Convert numeric CDC fields; keep Y/N/U fields as upper-case strings."""
    df["official_cdc_bmi"] = parse_cdc_bmi(df["official_cdc_bmi"])

    numeric_cols = [
        "mother_age", "mother_education_code",
        "prior_live_births", "prior_dead_births", "prior_other_terminations",
        "live_birth_order", "total_birth_order",
        "interval_since_last_live_birth_raw", "interval_since_last_pregnancy_raw",
        "prenatal_care_start_month", "prenatal_visits_count",
        "cigarettes_before_pregnancy", "cigarettes_trimester_1", "cigarettes_trimester_2", "cigarettes_trimester_3",
        "mother_height_inches", "pre_pregnancy_weight_lb", "gestational_weight_gain_lb",
        "previous_cesarean_count",
        "plurality_code", "gestational_age_weeks", "birth_weight_g",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["infant_sex", "wic_received"] + RISK_YN_COLS:
        df[col] = df[col].astype(str).str.strip().str.upper()

    return df

def assert_bmi_within_cdc_bounds(df: pd.DataFrame) -> None:
    """Integrity check, not a data-cleaning step.

    The User Guide guarantees official_cdc_bmi is already within
    13.0-69.9 (or NaN) in the published file. If this fails, it signals a
    parsing bug in our own pipeline -- check the BMI column offset or the
    decimal-point handling in parse_cdc_bmi before trusting this dataset.
    """
    out_of_bounds = df["official_cdc_bmi"].notna() & ~df["official_cdc_bmi"].between(BMI_MIN, BMI_MAX)
    n_out_of_bounds = int(out_of_bounds.sum())
    assert n_out_of_bounds == 0, (
        f"{n_out_of_bounds} BMI values outside the CDC-documented {BMI_MIN}-{BMI_MAX} range. "
        "This should be impossible per the User Guide -- check the BMI column "
        "offset or decimal-point parsing before trusting this dataset."
    )
    print(f"BMI integrity check passed: all {df['official_cdc_bmi'].notna().sum()} known BMI values are within {BMI_MIN}-{BMI_MAX}.")

## 5. CDC missing codes

CDC "unknown/not stated" sentinel codes are converted to `NaN` here, field
by field. The interval fields (`intervalo_*_raw`) are **not** touched here:
their `888` ("not applicable") sentinel is not the same thing as "unknown",
and is handled separately in step 8 so it never enters the model as a
numeric value of 888 months.`official_cdc_bmi` is **not** touched here either — its sentinel handling
already happened inside `parse_cdc_bmi`.

In [5]:
def replace_cdc_missing_codes(df: pd.DataFrame) -> pd.DataFrame:
    """Replace CDC unknown/not-stated codes with NaN.

    888 sentinel codes in interval fields are deliberately NOT replaced
    here: they mean "not applicable", not "unknown" (handled in step 8).
    """
    for col in ["cigarettes_before_pregnancy", "cigarettes_trimester_1", "cigarettes_trimester_2", "cigarettes_trimester_3"]:
        df.loc[df[col] == 99, col] = np.nan

    for col in ["prior_live_births", "prior_dead_births", "prior_other_terminations"]:
        df.loc[df[col] == 99, col] = np.nan
    for col in ["live_birth_order", "total_birth_order"]:
        df.loc[df[col] == 9, col] = np.nan

    df.loc[df["mother_height_inches"] == 99, "mother_height_inches"] = np.nan
    df.loc[df["pre_pregnancy_weight_lb"] == 999, "pre_pregnancy_weight_lb"] = np.nan
    df.loc[df["birth_weight_g"] == 9999, "birth_weight_g"] = np.nan
    df.loc[df["gestational_age_weeks"] == 99, "gestational_age_weeks"] = np.nan
    df.loc[df["mother_education_code"] == 9, "mother_education_code"] = np.nan

    df.loc[df["gestational_weight_gain_lb"] == 99, "gestational_weight_gain_lb"] = np.nan
    df.loc[df["prenatal_care_start_month"] == 99, "prenatal_care_start_month"] = np.nan
    df.loc[df["prenatal_visits_count"] == 99, "prenatal_visits_count"] = np.nan
    df.loc[df["previous_cesarean_count"] == 99, "previous_cesarean_count"] = np.nan

    return df

## 6. Row-level filters

Only five filters are applied, all aimed at removing CDC-documented
"unknown" sentinel rows or rows incompatible with the study design
(multiples). 

In [6]:
def apply_basic_filters(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply only the row filters needed for this study design."""
    steps = []

    def keep(data: pd.DataFrame, mask, label: str) -> pd.DataFrame:
        out = data.loc[mask].copy()
        steps.append({
            "step": label,
            "rows_remaining": int(len(out)),
            "rows_removed": int(len(data) - len(out)),
        })
        return out

    steps.append({"step": "raw", "rows_remaining": int(len(df)), "rows_removed": 0})

    df = keep(df, df["plurality_code"].eq(1), "singletons only")
    df = keep(df, df["birth_weight_g"].between(BIRTH_WEIGHT_MIN, BIRTH_WEIGHT_MAX),
               f"CDC edited birthweight range {BIRTH_WEIGHT_MIN}-{BIRTH_WEIGHT_MAX} g")
    df = keep(df, df["mother_age"].ge(MOTHER_AGE_MIN), f"maternal age >= {MOTHER_AGE_MIN}")
    df = keep(df, df["infant_sex"].isin(["M", "F"]), "known fetal sex")
    df = keep(df, df["gestational_age_weeks"].notna(), "known combined gestational age")

    return df, pd.DataFrame(steps)

## 7. Education and body-size features

Maternal education is kept as the numeric code plus a missing flag, no
recode lookup is reproduced.

`bmi_calculado` keeps the name expected by the existing `conformal_core.py`
downstream code, even though every other column in this notebook now uses
readable names — this is the one deliberate exception, purely for
compatibility with code outside this notebook.

In [7]:
def encode_education(df: pd.DataFrame) -> pd.DataFrame:
    """Keep maternal education code and a missing flag, no recode lookup."""
    df["mother_education_missing"] = df["mother_education_code"].isna().astype(int)
    return df


def derive_body_features(df: pd.DataFrame) -> pd.DataFrame:
    """Body-size features. official_cdc_bmi is never overwritten based on range."""
    df["mother_height_missing"] = df["mother_height_inches"].isna().astype(int)
    df["pre_pregnancy_weight_missing"] = df["pre_pregnancy_weight_lb"].isna().astype(int)
    df["official_cdc_bmi_missing"] = df["official_cdc_bmi"].isna().astype(int)

    df["mother_height_cm"] = (df["mother_height_inches"] * 2.54).round(2)
    df["pre_pregnancy_weight_kg"] = (df["pre_pregnancy_weight_lb"] * 0.453592).round(2)

    # Name kept for compatibility with conformal_core.py.
    df["bmi_calculado"] = df["official_cdc_bmi"].round(2)

    return df

## 8. Smoking features

CDC code `98` means *"98 or more cigarettes/day"* and is kept as `98`
(flagged separately via `smoked_98plus_any_period`) rather than clipped
to `97`, so the heaviest-smoking observations are not silently truncated.

`smoking_intensity_group` is the 4-level smoking-intensity axis used
downstream in the preterm × smoking interaction:

$$
\text{smoking\_intensity\_group} \in \{\text{Non-smoker}, \text{Low (1-5)}, \text{Moderate (6-10)}, \text{Heavy (>10)}\}
$$

In [8]:
SMOKING_BINS = [-0.001, 0, 5, 10, 200]
SMOKING_LABELS = [
    "Non-smoker (0 cig/day)",
    "Low smoker (1-5 cig/day)",
    "Moderate smoker (6-10 cig/day)",
    "Heavy smoker (>10 cig/day)",
]


def derive_smoking_features(df: pd.DataFrame) -> pd.DataFrame:
    """Impute cigarette variables and build the smoking-intensity axis."""
    cig_cols = ["cigarettes_before_pregnancy"] + PREGNANCY_CIG_COLS
    for col in cig_cols:
        df[f"{col}_missing"] = df[col].isna().astype(int)

    # Must be computed before imputation: checks whether all three trimester
    # fields were NaN, which fillna() below would otherwise erase any trace of.
    df["pregnancy_cig_all_missing"] = df[PREGNANCY_CIG_COLS].isna().all(axis=1).astype(int)

    df["avg_cig_pregnancy_available"] = df[PREGNANCY_CIG_COLS].mean(axis=1, skipna=True)
    for col in PREGNANCY_CIG_COLS:
        df[col] = df[col].fillna(df["avg_cig_pregnancy_available"]).fillna(0)
    df["cigarettes_before_pregnancy"] = df["cigarettes_before_pregnancy"].fillna(0)

    for col in cig_cols:
        df[col] = df[col].clip(lower=0, upper=98)

    df["smoked_98plus_any_period"] = (df[cig_cols] == 98).any(axis=1).astype(int)
    df["avg_cig_pregnancy"] = df[PREGNANCY_CIG_COLS].mean(axis=1).round(2)
    df["max_cig_pregnancy"] = df[PREGNANCY_CIG_COLS].max(axis=1)
    df["smoked_during_pregnancy"] = (df["max_cig_pregnancy"] > 0).astype(int)
    df["smoked_before_pregnancy"] = (df["cigarettes_before_pregnancy"] > 0).astype(int)

    df["smoking_intensity_group"] = pd.cut(df["avg_cig_pregnancy"], bins=SMOKING_BINS, labels=SMOKING_LABELS)

    return df

## 9. Weight gain, prenatal care and WIC

`gestational_weight_gain_lb` is kept numeric with a missing flag and a
`98plus` flag (same "don't truncate the extreme" logic as smoking).

For prenatal care, `no_prenatal_visits` and
`few_prenatal_visits` are mutually exclusive:

- `no_prenatal_visits`: exactly 0 visits.
- `few_prenatal_visits`: 1 to 4 visits.

In [9]:
def derive_weight_gain_features(df: pd.DataFrame) -> pd.DataFrame:
    """Pregnancy weight gain. Kept numeric; CDC recode bins not reproduced."""
    df["gestational_weight_gain_missing"] = df["gestational_weight_gain_lb"].isna().astype(int)
    df["gestational_weight_gain_98plus"] = (df["gestational_weight_gain_lb"] == 98).astype(int)
    return df


def derive_prenatal_wic_features(df: pd.DataFrame) -> pd.DataFrame:
    """Prenatal care and WIC, derived directly from raw fields.

    no_prenatal_visits and few_prenatal_visits are mutually
    exclusive: 0 visits vs. 1-4 visits.
    """
    df["prenatal_care_start_month_missing"] = df["prenatal_care_start_month"].isna().astype(int)
    df["prenatal_visits_count_missing"] = df["prenatal_visits_count"].isna().astype(int)

    df["no_prenatal_care"] = (df["prenatal_care_start_month"] == 0).astype(int)
    df["early_prenatal_care"] = df["prenatal_care_start_month"].between(1, 3).astype(int)
    df["late_prenatal_care"] = df["prenatal_care_start_month"].ge(7).astype(int)

    df["no_prenatal_visits"] = (df["prenatal_visits_count"] == 0).astype(int)
    df["few_prenatal_visits"] = df["prenatal_visits_count"].between(1, 4).astype(int)

    wic = df["wic_received"].astype(str).str.strip().str.upper()
    df["wic_bin"] = wic.map({"Y": 1, "N": 0})
    df["wic_missing"] = df["wic_bin"].isna().astype(int)
    df["wic_bin"] = df["wic_bin"].fillna(0).astype(int)

    return df

## 10. Risk factors and previous cesarean (with the inconsistency flag)

Risk factors are mapped to binary + missing-flag pairs, same pattern used
throughout this project.

For previous cesarean, `cesarean_history_mismatch` surfaces a genuine CDC
record-level contradiction — `previous_cesarean` (Y/N) disagreeing with
`previous_cesarean_count` (count) — instead of silently resolving it with
an `OR`.

In [10]:
def yn_to_binary(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Map Y/N/U risk variables to binary + missing flag."""
    s = series.astype(str).str.strip().str.upper()
    mapped = s.map({"Y": 1, "N": 0})
    missing = mapped.isna().astype(int)
    return mapped.fillna(0).astype(int), missing


def derive_risk_features(df: pd.DataFrame) -> pd.DataFrame:
    """Encode maternal risk factors, infertility treatment and previous cesarean."""
    for col in RISK_YN_COLS:
        df[f"{col}_bin"], df[f"{col}_missing"] = yn_to_binary(df[col])

    df["previous_cesarean_count_missing"] = df["previous_cesarean_count"].isna().astype(int)
    df["previous_cesarean_count"] = df["previous_cesarean_count"].fillna(0).clip(lower=0, upper=30).astype(int)

    df["has_previous_cesarean"] = (
        (df["previous_cesarean_bin"] == 1) | (df["previous_cesarean_count"] > 0)
    ).astype(int)

    # Explicit flag for CDC record-level contradictions, instead of a silent OR.
    df["cesarean_history_mismatch"] = (
        (df["previous_cesarean_bin"] != (df["previous_cesarean_count"] > 0).astype(int))
        & (df["previous_cesarean_missing"] == 0)
    ).astype(int)

    return df

## 11. Obstetric history and pregnancy intervals

Parity counts and birth-order recodes are imputed with flags, same pattern
as before.

For the interval fields, the CDC sentinel `888` ("not applicable", e.g.
first live birth) is kept separate from both the real numeric value and
from genuine missingness:

- `*_meses`: the real number of months, only when it's a valid value
  (`4`–`300`). Otherwise `NaN`.
- `*_not_applicable`: `1` when the original code was `888`.
- `*_missing`: `1` when the value is genuinely unknown (`999` or blank).

In [11]:
def add_interval_feature(df: pd.DataFrame, raw_col: str, prefix: str) -> pd.DataFrame:
    """Handle a CDC 3-digit interval field without treating 888 as a numeric outlier.

    004-300: valid number of months.
    888: not applicable (e.g. first live birth / no previous pregnancy).
    999: unknown.
    """
    x = pd.to_numeric(df[raw_col], errors="coerce")
    df[f"{prefix}_meses"] = x.where(x.between(4, 300), np.nan)
    df[f"{prefix}_not_applicable"] = (x == 888).astype(int)
    df[f"{prefix}_missing"] = (x.isna() | (x == 999)).astype(int)
    return df


def derive_obstetric_history_features(df: pd.DataFrame) -> pd.DataFrame:
    """Impute obstetric history counts and derive parity-related features."""
    for col in ["prior_live_births", "prior_dead_births",
                "prior_other_terminations", "live_birth_order", "total_birth_order"]:
        df[f"{col}_missing"] = df[col].isna().astype(int)

    for col in ["prior_live_births", "prior_dead_births", "prior_other_terminations"]:
        df[col] = df[col].fillna(0).clip(lower=0, upper=30)

    for col in ["live_birth_order", "total_birth_order"]:
        df[col] = df[col].fillna(df[col].median()).fillna(1).clip(lower=1, upper=8)

    df["has_prior_live_birth"] = (df["prior_live_births"] > 0).astype(int)
    df["has_prior_dead_birth"] = (df["prior_dead_births"] > 0).astype(int)
    df["has_prior_other_termination"] = (df["prior_other_terminations"] > 0).astype(int)
    df["total_prior_pregnancy_history"] = (
        df["prior_live_births"] + df["prior_dead_births"] + df["prior_other_terminations"]
    ).astype(int)
    df["is_first_live_birth"] = (df["live_birth_order"] == 1).astype(int)
    df["is_first_total_pregnancy"] = (df["total_birth_order"] == 1).astype(int)

    df = add_interval_feature(df, "interval_since_last_live_birth_raw", "interval_since_last_live_birth")
    df = add_interval_feature(df, "interval_since_last_pregnancy_raw", "interval_since_last_pregnancy")

    return df

## 12. Target variable and the preterm × smoking evaluation axis

`baby_poids_kg` is the regression target (name kept for compatibility with
`conformal_core.py`). `preterm_current` follows the ICD-9/ICD-10 definition
used by CDC itself (`gestational_age_weeks < 37`).

`preterm_x_smoking_intensity` is the single evaluation axis agreed for this
study, persisted as an explicit column:

$$
\text{preterm\_x\_smoking\_intensity} = \{\text{preterm}, \text{term}\} \times \{\text{4 smoking levels}\} = \text{8 groups}
$$

No group is merged or dropped here for having a small sample size.

In [12]:
def derive_target_and_groups(df: pd.DataFrame) -> pd.DataFrame:
    """Target, LBW flag, preterm flag, fetal sex."""
    df["infant_sex"] = df["infant_sex"].where(df["infant_sex"].isin(["M", "F"]), np.nan)
    df["is_male"] = df["infant_sex"].map({"M": 1, "F": 0})

    # Names kept for compatibility with conformal_core.py.
    df["baby_poids_kg"] = (df["birth_weight_g"] / 1000).round(3)
    df["LBW"] = (df["baby_poids_kg"] < 2.5).astype(int)
    df["preterm_current"] = (df["gestational_age_weeks"] < 37).astype(int)

    return df


def derive_preterm_x_smoking(df: pd.DataFrame) -> pd.DataFrame:
    """Build and persist the preterm x smoking-intensity evaluation axis.

    2 preterm levels x 4 smoking-intensity levels = 8 groups. No group is
    merged or dropped for small n here; per-group n is reported separately
    in show_diagnostics.
    """
    preterm_label = np.where(df["preterm_current"] == 1, "preterm", "term")
    df["preterm_x_smoking_intensity"] = (
        pd.Series(preterm_label, index=df.index).astype(str) + " | " + df["smoking_intensity_group"].astype(str)
    )
    return df

## 13. Final column selection

Only target, model features, the evaluation axis, and diagnostic flags are
kept. No race/ethnicity columns, no CDC recode columns. `bmi_calculado`,
`baby_poids_kg`, `LBW`, and `preterm_current` keep their original names for
compatibility with `conformal_core.py`; everything else uses the readable
names introduced in this notebook.

In [13]:
def final_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Keep target, model features, the evaluation axis and diagnostics."""
    cols = [
        # Target and outcome descriptors
        "baby_poids_kg", "LBW", "gestational_age_weeks", "preterm_current", "plurality_code",

        # Evaluation / Mondrian axis
        "preterm_x_smoking_intensity",

        # Maternal age and education
        "mother_age", "mother_education_code", "mother_education_missing",

        # Anthropometrics
        "mother_height_inches", "mother_height_missing", "mother_height_cm",
        "pre_pregnancy_weight_lb", "pre_pregnancy_weight_missing", "pre_pregnancy_weight_kg",
        "bmi_calculado", "official_cdc_bmi_missing",

        # Fetal sex
        "infant_sex", "is_male",

        # Smoking
        "cigarettes_before_pregnancy", "cigarettes_trimester_1", "cigarettes_trimester_2", "cigarettes_trimester_3",
        "cigarettes_before_pregnancy_missing", "cigarettes_trimester_1_missing",
        "cigarettes_trimester_2_missing", "cigarettes_trimester_3_missing",
        "smoked_98plus_any_period", "pregnancy_cig_all_missing",
        "avg_cig_pregnancy_available", "avg_cig_pregnancy", "max_cig_pregnancy",
        "smoking_intensity_group", "smoked_before_pregnancy", "smoked_during_pregnancy",

        # Pregnancy weight gain
        "gestational_weight_gain_lb", "gestational_weight_gain_missing", "gestational_weight_gain_98plus",

        # Prenatal care and WIC
        "prenatal_care_start_month", "prenatal_care_start_month_missing",
        "no_prenatal_care", "early_prenatal_care", "late_prenatal_care",
        "prenatal_visits_count", "prenatal_visits_count_missing",
        "no_prenatal_visits", "few_prenatal_visits",
        "wic_received", "wic_bin", "wic_missing",

        # Obstetric history and parity
        "prior_live_births", "prior_dead_births", "prior_other_terminations",
        "live_birth_order", "total_birth_order",
        "prior_live_births_missing", "prior_dead_births_missing",
        "prior_other_terminations_missing", "live_birth_order_missing", "total_birth_order_missing",
        "has_prior_live_birth", "has_prior_dead_birth", "has_prior_other_termination",
        "total_prior_pregnancy_history", "is_first_live_birth", "is_first_total_pregnancy",
        "interval_since_last_live_birth_months", "interval_since_last_live_birth_not_applicable", "interval_since_last_live_birth_missing",
        "interval_since_last_pregnancy_months", "interval_since_last_pregnancy_not_applicable", "interval_since_last_pregnancy_missing",

        # Risk factors
        "pre_pregnancy_diabetes_bin", "gestational_diabetes_bin", "pre_pregnancy_hypertension_bin",
        "gestational_hypertension_bin", "eclampsia_bin", "previous_preterm_birth_bin",
        "pre_pregnancy_diabetes_missing", "gestational_diabetes_missing", "pre_pregnancy_hypertension_missing",
        "gestational_hypertension_missing", "eclampsia_missing", "previous_preterm_birth_missing",
        "infertility_treatment_bin", "infertility_treatment_missing",
        "previous_cesarean_bin", "previous_cesarean_missing",
        "previous_cesarean_count", "previous_cesarean_count_missing",
        "has_previous_cesarean", "cesarean_history_mismatch",
    ]
    return df[[c for c in cols if c in df.columns]].copy()

## 14. Diagnostics

`report_preterm_x_smoking_counts` prints the `n` per group of the evaluation
axis, sorted alphabetically, with no merging.

`show_diagnostics` also reports how many rows were flagged by
`cesarean_history_mismatch`, so you can see whether this is a rare edge case
or something that needs more attention. The BMI integrity check from step
4.1 already ran earlier in the pipeline and would have stopped execution if
it failed.

In [14]:
def report_preterm_x_smoking_counts(df: pd.DataFrame) -> pd.DataFrame:
    """n per group of the preterm x smoking-intensity axis, sorted by group."""
    counts = df["preterm_x_smoking_intensity"].value_counts(dropna=False).sort_index()
    out = counts.to_frame("n")
    out["pct_of_total"] = (out["n"] / len(df) * 100).round(3)
    return out


def show_diagnostics(df: pd.DataFrame, filter_log: pd.DataFrame) -> None:
    print("\n" + "=" * 80)
    print("FILTER LOG")
    print("=" * 80)
    log = filter_log.assign(**{"% of raw": (filter_log["rows_remaining"] / filter_log.loc[0, "rows_remaining"] * 100).round(2)})
    display(log)

    print("\n" + "=" * 80)
    print("FINAL SHAPE AND KEY COUNTS")
    print("=" * 80)
    print(f"Shape: {df.shape}")
    print(f"Birthweight kg: min={df['baby_poids_kg'].min():.3f}, max={df['baby_poids_kg'].max():.3f}")
    print(f"Preterm: {df['preterm_current'].sum()} ({df['preterm_current'].mean()*100:.2f}%)")
    print(f"Smoking during pregnancy: {df['smoked_during_pregnancy'].sum()} ({df['smoked_during_pregnancy'].mean()*100:.2f}%)")

    print("\n" + "=" * 80)
    print("PRETERM x SMOKING INTENSITY -- n per group (no merging applied)")
    print("=" * 80)
    display(report_preterm_x_smoking_counts(df))

    print("\n" + "=" * 80)
    print("DIAGNOSTIC FLAGS")
    print("=" * 80)
    print(f"cesarean_history_mismatch (previous_cesarean vs previous_cesarean_count disagree): "
          f"{int(df['cesarean_history_mismatch'].sum())} ({df['cesarean_history_mismatch'].mean()*100:.3f}%)")

    print("\n" + "=" * 80)
    print("TOP MISSING COLUMNS")
    print("=" * 80)
    display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("NaN count"))

## 15. Complete pipeline

This ties every step together in order, including the BMI integrity check
right after numeric conversion, and writes both the cleaned CSV and the
filter log to disk.

In [15]:
def build_clean_dataset(raw_file: str = FILE_PATH, output_csv: str = OUTPUT_CSV) -> pd.DataFrame:
    """Run the complete cleaning pipeline and save the final CSV."""
    raw_path = Path(raw_file)
    out_path = Path(output_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    filter_log_path = Path(OUTPUT_FILTER_LOG) if OUTPUT_FILTER_LOG else out_path.with_name(out_path.stem + "_filter_log.csv")

    print(f"Reading raw file: {raw_path}")
    df = read_raw_birth_file(str(raw_path))
    print(f"Raw shape: {df.shape}")

    df = to_numeric_columns(df)
    assert_bmi_within_cdc_bounds(df)
    df = replace_cdc_missing_codes(df)
    df, filter_log = apply_basic_filters(df)

    df = encode_education(df)
    df = derive_body_features(df)
    df = derive_smoking_features(df)
    df = derive_weight_gain_features(df)
    df = derive_prenatal_wic_features(df)
    df = derive_risk_features(df)
    df = derive_obstetric_history_features(df)
    df = derive_target_and_groups(df)
    df = derive_preterm_x_smoking(df)

    final = final_columns(df)

    final.to_csv(out_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
    filter_log.to_csv(filter_log_path, index=False)

    print(f"\nSaved clean CSV: {out_path}")
    print(f"Saved filter log: {filter_log_path}")
    print(f"Final shape: {final.shape}")

    show_diagnostics(final, filter_log)
    return final

## 16. Run

Check `FILE_PATH` in the configuration cell (step 1) before running this.

In [16]:
final = build_clean_dataset(FILE_PATH, OUTPUT_CSV)
final.head()

Reading raw file: Birth2022(1000k).txt
Raw shape: (1000000, 33)
BMI integrity check passed: all 975020 known BMI values are within 13.0-69.9.

Saved clean CSV: natality2022_clean_preterm_smoking.csv
Saved filter log: natality2022_clean_preterm_smoking_filter_log.csv
Final shape: (968459, 90)

FILTER LOG


,step,rows_remaining,rows_removed,% of raw
0,raw,1000000,0,100.00
1,singletons only,969528,30472,96.95
2,CDC edited birthweight range 227-8165 g,968953,575,96.90
3,maternal age >= 10,968953,0,96.90
4,known fetal sex,968953,0,96.90
5,known combined gestational age,968459,494,96.85



FINAL SHAPE AND KEY COUNTS
Shape: (968459, 90)
Birthweight kg: min=0.227, max=6.747
Preterm: 101133 (10.44%)
Smoking during pregnancy: 20129 (2.08%)

PRETERM x SMOKING INTENSITY -- n per group (no merging applied)


,n,pct_of_total
preterm_x_smoking_intensity,,
preterm | Heavy smoker (>10 cig/day),799,0.083
preterm | Low smoker (1-5 cig/day),1591,0.164
preterm | Moderate smoker (6-10 cig/day),1247,0.129
preterm | Non-smoker (0 cig/day),97496,10.067
term | Heavy smoker (>10 cig/day),3330,0.344
term | Low smoker (1-5 cig/day),7773,0.803
term | Moderate smoker (6-10 cig/day),5389,0.556
term | Non-smoker (0 cig/day),850834,87.854



DIAGNOSTIC FLAGS
cesarean_history_mismatch (previous_cesarean vs previous_cesarean_count disagree): 75 (0.008%)

TOP MISSING COLUMNS


,NaN count
mother_education_code,36771
gestational_weight_gain_lb,27504
bmi_calculado,23799
pre_pregnancy_weight_lb,21852
pre_pregnancy_weight_kg,21852
prenatal_visits_count,18097
prenatal_care_start_month,17795
mother_height_cm,4050
mother_height_inches,4050
avg_cig_pregnancy_available,2780


,baby_poids_kg,LBW,gestational_age_weeks,preterm_current,plurality_code,preterm_x_smoking_intensity,mother_age,mother_education_code,mother_education_missing,mother_height_inches,...,eclampsia_missing,previous_preterm_birth_missing,infertility_treatment_bin,infertility_treatment_missing,previous_cesarean_bin,previous_cesarean_missing,previous_cesarean_count,previous_cesarean_count_missing,has_previous_cesarean,cesarean_history_mismatch
0,3.203,0,42.0,0,1,term | Non-smoker (0 cig/day),37,6.0,0,67.0,...,0,0,0,0,0,0,0,0,0,0
1,3.090,0,38.0,0,1,term | Non-smoker (0 cig/day),27,3.0,0,61.0,...,0,0,0,0,0,0,0,0,0,0
2,2.523,0,37.0,0,1,term | Non-smoker (0 cig/day),25,5.0,0,68.0,...,0,0,0,0,0,0,0,0,0,0
3,4.139,0,38.0,0,1,term | Non-smoker (0 cig/day),42,7.0,0,67.0,...,0,0,0,0,0,0,0,0,0,0
4,3.000,0,37.0,0,1,term | Non-smoker (0 cig/day),27,3.0,0,65.0,...,0,0,0,0,0,0,0,0,0,0
